In [ ]:
%%writefile dgemm.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

// The Algorithm: Double precision, General Matrix Multiply
void dgemm (size_t n, double* A, double* B, double* C) {
    for (size_t i = 0; i < n; ++i)
        for (size_t j = 0; j < n; ++j) {
            double cij = C[i+j*n]; 
            for(size_t k = 0; k < n; k++)
                cij += A[i+k*n] * B[k+j*n]; 
            C[i+j*n] = cij; 
        }
}

int main() {
    time_t start, end;
    const int rowSize = 1024; // Matrix size
    srand(123);

    // Allocate memory
    double *a = (double *)malloc(rowSize * rowSize * sizeof(double));
    double *b = (double *)malloc(rowSize * rowSize * sizeof(double));
    double *c = (double *)malloc(rowSize * rowSize * sizeof(double));

    // Initialize matrices with random values
    for(int i=0; i< rowSize; i++) {
        for(int j=0; j< rowSize; j++) {
            a[i+j*rowSize] = rand() / (double)RAND_MAX;
            b[i+j*rowSize] = rand() / (double)RAND_MAX;
            c[i+j*rowSize] = 0;
        }
    }

    // Run benchmark
    start = clock();
    dgemm(rowSize, c, a, b);
    end = clock();

    // Clean up
    if(a) free(a); 
    if(b) free(b); 
    if(c) free(c);
    
    // Calculate actual time in milliseconds
    double time_taken = ((double)(end - start)) / CLOCKS_PER_SEC * 1000;
    printf("Execution Time: %f ms\n", time_taken);
    return 0;
}

In [ ]:
%%bash
echo "📊 Benchmarking: GCC No Optimization (-O0)"
gcc -O0 dgemm.c -o dgemm_o0
./dgemm_o0

echo "----------------------------------------"
echo "📊 Benchmarking: GCC High Optimization (-O3)"
gcc -O3 dgemm.c -o dgemm_o3
./dgemm_o3

echo "----------------------------------------"
echo "📊 Benchmarking: GCC + Auto-Vectorization (-O3 -march=native)"
gcc -O3 -march=native dgemm.c -o dgemm_fast
./dgemm_fast